# Semana 12 · Diseño de una solución empresarial con PLN
## Caso: análisis automático de reseñas de viajeros (*Traveler Insights UPTC 2026*)

**Curso:** Procesamiento de Lenguaje Natural · Universidad Tecnológica de Bolívar
**Unidad 4:** Aplicaciones avanzadas y evaluación de sistemas de PLN

Este notebook muestra cómo pasar de **un modelo que clasifica texto** a **una solución que una organización podría operar**. Usa los datos del proyecto integrador y recorre la cadena completa:

**problema de negocio → datos → baseline → modelo → análisis de errores → aspectos → respuesta automática → humano en el bucle → impacto → reflexión crítica**

### Resultados de aprendizaje
- **RDA1.** Describir los principios de funcionamiento de un sistema de PLN de punta a punta.
- **RDA2.** Analizar el impacto de una aplicación de PLN en un entorno empresarial (costo, eficiencia, experiencia del usuario).
- **RDA3.** Diseñar e implementar los componentes algorítmicos de un sistema inteligente basado en PLN.

### Antes de empezar
1. Descarga `train.csv` de la competencia de Kaggle *Traveler Insights UPTC 2026*.
2. Ajusta la ruta en la celda de **Configuración**.
3. Si conoces los nombres de las columnas de texto y etiqueta, escríbelos; si no, el notebook intenta detectarlos.

> Las secciones con modelo Transformer y LLM local son **opcionales** (requieren GPU o paciencia). El resto corre en CPU en pocos minutos.

## 0. Configuración

In [ ]:
# ====== CONFIGURACIÓN (edita solo esta celda) ======
DATA_PATH   = "data/train.csv"   # ruta al archivo de entrenamiento de Kaggle
TEXT_COL    = None               # p. ej. "review"; None = detección automática
LABEL_COL   = None               # p. ej. "label";  None = detección automática
RANDOM_STATE = 42
VALID_SIZE   = 0.2

# Opcional 1: tu Transformer afinado en el proyecto (carpeta local o id de Hugging Face)
USE_TRANSFORMER = False
TRANSFORMER_PATH = "ruta/a/tu/modelo_afinado"

# Opcional 2: LLM local open source para redactar respuestas
USE_LLM = False
LLM_ID  = "Qwen/Qwen2.5-0.5B-Instruct"   # modelo pequeño, corre en CPU (lento) o GPU

In [ ]:
import re, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
plt.rcParams["figure.dpi"] = 110

## 1. Del problema de negocio al problema de PLN

Antes de escribir código, una solución empresarial se especifica. Usamos el **canvas de solución con PLN** visto en clase. El escenario es **hipotético** y sirve de guía: reemplázalo por el de tu equipo.

| Elemento | Escenario guía |
|---|---|
| **Organización** | Operador turístico regional que recibe reseñas de viajeros en varios canales. |
| **Dolor actual** | Las reseñas se leen a mano, tarde y sin agregación: nadie sabe qué aspecto (limpieza, servicio, precio…) genera más quejas. |
| **Tarea de PLN** | Clasificación de texto (la etiqueta del dataset) + detección de aspectos + redacción asistida de respuestas. |
| **Datos** | Reseñas del dataset Traveler Insights (texto + etiqueta). |
| **Baseline** | TF-IDF + Regresión logística. |
| **Modelo** | Transformer afinado del proyecto integrador. |
| **Métrica técnica** | Macro-F1 (las clases suelen estar desbalanceadas). |
| **Métrica de negocio** | % de reseñas atendidas sin intervención, horas ahorradas, tiempo de respuesta. |
| **Riesgos** | Errores en reseñas negativas, respuestas que prometan algo indebido, sesgo dialectal, privacidad. |
| **Humano en el bucle** | Las predicciones con baja confianza y las respuestas sensibles las revisa una persona. |

**Pregunta guía:** *¿qué decisión de la organización cambia gracias a este sistema?* Si no hay respuesta, no hay caso de negocio.

## 2. Carga de datos y detección de columnas

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print("Dimensiones:", df_raw.shape)
display(df_raw.head())
print(df_raw.dtypes)

In [ ]:
def detectar_columnas(df, text_col=None, label_col=None):
    # Texto: columna de tipo cadena con mayor longitud media.
    # Etiqueta: columna (distinta del texto) con pocas categorías (2..20).
    obj_cols = [c for c in df.columns
                if pd.api.types.is_string_dtype(df[c]) or pd.api.types.is_object_dtype(df[c])]
    if text_col is None:
        if not obj_cols:
            raise ValueError("No hay columnas de texto; define TEXT_COL.")
        text_col = max(obj_cols, key=lambda c: df[c].astype(str).str.len().mean())
    if label_col is None:
        candidatas = [c for c in df.columns
                      if c != text_col and 2 <= df[c].nunique() <= 20
                      and not str(c).lower().startswith("id")]
        if not candidatas:
            raise ValueError("No encontré una columna de etiqueta; define LABEL_COL.")
        label_col = min(candidatas, key=lambda c: df[c].nunique())
    return text_col, label_col

TEXT_COL, LABEL_COL = detectar_columnas(df_raw, TEXT_COL, LABEL_COL)
print(f"Columna de texto:    {TEXT_COL}")
print(f"Columna de etiqueta: {LABEL_COL}")

### 2.1 Calidad de datos
En un proyecto real, **la mitad de los problemas está aquí**: vacíos, duplicados y desbalance.

In [ ]:
df = df_raw[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "texto", LABEL_COL: "etiqueta"})
n0 = len(df)
vacios = df["texto"].isna().sum() + (df["texto"].astype(str).str.strip() == "").sum()
df = df.dropna(subset=["texto", "etiqueta"])
df["texto"] = df["texto"].astype(str)
df = df[df["texto"].str.strip() != ""]
dup = df.duplicated(subset=["texto"]).sum()
conflictos = df.groupby("texto")["etiqueta"].nunique().gt(1).sum()
df = df.drop_duplicates(subset=["texto"]).reset_index(drop=True)
df["etiqueta"] = df["etiqueta"].astype(str)

print(f"Filas originales: {n0} | vacías: {vacios} | textos duplicados: {dup} "
      f"| textos con etiquetas contradictorias: {conflictos} | filas finales: {len(df)}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
dist = df["etiqueta"].value_counts()
dist.plot.bar(ax=ax[0], color="#0B1F5C")
ax[0].set_title("Distribución de clases"); ax[0].set_ylabel("reseñas")
df["n_palabras"] = df["texto"].str.split().str.len()
df["n_palabras"].clip(upper=df["n_palabras"].quantile(0.99)).plot.hist(bins=40, ax=ax[1], color="#2E6FD8")
ax[1].set_title("Longitud de las reseñas (palabras)")
plt.tight_layout(); plt.show()

print((dist / dist.sum()).round(3).to_string())
print("Razón de desbalance (mayor/menor):", round(dist.max() / dist.min(), 1))

**Para discutir:** si la razón de desbalance es alta, la *accuracy* engaña. Por eso reportamos **macro-F1**, que pesa igual a cada clase.

## 3. Partición y control de fuga de información
El `test.csv` de Kaggle no trae etiquetas, así que validamos con una partición estratificada de `train.csv`.

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(
    df["texto"], df["etiqueta"], test_size=VALID_SIZE,
    stratify=df["etiqueta"], random_state=RANDOM_STATE)

def normalizar(t):
    t = unicodedata.normalize("NFKD", t.lower())
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", t).strip()

fuga = len(set(X_tr.map(normalizar)) & set(X_va.map(normalizar)))
print(f"Entrenamiento: {len(X_tr)} | Validación: {len(X_va)}")
print(f"Textos casi idénticos presentes en ambos conjuntos (fuga): {fuga}")

## 4. Baselines: ¿cuánto vale *no* usar un Transformer?
Un caso de negocio honesto compara contra lo más simple que funciona:
1. **Dummy:** predice siempre la clase mayoritaria (piso mínimo).
2. **TF-IDF + Naive Bayes:** el clásico de Jurafsky & Martin (cap. 4).
3. **TF-IDF palabras + caracteres + Regresión logística:** baseline fuerte; los n-gramas de caracteres toleran errores ortográficos y variación dialectal.

In [ ]:
modelos = {
    "Dummy (mayoritaria)": DummyClassifier(strategy="most_frequent"),
    "TF-IDF + NaiveBayes": Pipeline([
        ("tfidf", TfidfVectorizer(preprocessor=normalizar, ngram_range=(1, 2), min_df=2)),
        ("clf", MultinomialNB())]),
    "TF-IDF (pal+car) + LogReg": Pipeline([
        ("feats", FeatureUnion([
            ("pal", TfidfVectorizer(preprocessor=normalizar, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
            ("car", TfidfVectorizer(preprocessor=normalizar, analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True))])),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=4.0))]),
}

resultados = []
for nombre, m in modelos.items():
    m.fit(X_tr, y_tr)
    p = m.predict(X_va)
    resultados.append({"modelo": nombre,
                       "accuracy": accuracy_score(y_va, p),
                       "macro_F1": f1_score(y_va, p, average="macro")})
tabla = pd.DataFrame(resultados).set_index("modelo").round(3)
tabla

In [ ]:
mejor = modelos["TF-IDF (pal+car) + LogReg"]
pred_va = mejor.predict(X_va)
print(classification_report(y_va, pred_va, digits=3))
ConfusionMatrixDisplay.from_predictions(y_va, pred_va, cmap="Blues", xticks_rotation=45)
plt.title("Matriz de confusión · baseline LogReg"); plt.tight_layout(); plt.show()

## 5. (Opcional) Comparar con el Transformer del proyecto
Activa `USE_TRANSFORMER = True` y apunta `TRANSFORMER_PATH` al modelo que afinaste. Se evalúa sobre **la misma partición**, para que la comparación sea justa.

> Verifica que los nombres de etiqueta del modelo (`id2label`) coincidan con los del dataset; si el modelo devuelve `LABEL_0`, `LABEL_1`…, ajusta el diccionario `mapa_etiquetas`.

In [ ]:
if USE_TRANSFORMER:
    from transformers import pipeline
    clf_tf = pipeline("text-classification", model=TRANSFORMER_PATH,
                      truncation=True, max_length=256)
    mapa_etiquetas = {}   # p. ej. {"LABEL_0": "negativo", "LABEL_1": "positivo"}
    salida = clf_tf(list(X_va), batch_size=32)
    pred_tf = [mapa_etiquetas.get(o["label"], o["label"]) for o in salida]
    tabla.loc["Transformer afinado"] = [accuracy_score(y_va, pred_tf),
                                        f1_score(y_va, pred_tf, average="macro")]
    display(tabla.round(3))
else:
    print("Sección omitida (USE_TRANSFORMER = False).")

**Pregunta de negocio:** si el Transformer mejora el macro-F1 en pocos puntos, ¿justifica su costo de inferencia, latencia y mantenimiento frente al baseline? La respuesta depende del **costo de cada error**, no solo de la métrica.

## 6. Análisis de errores: ¿*dónde* y *por qué* falla?
Un número agregado no dice qué arreglar. Miramos (a) los errores con mayor confianza y (b) las palabras que más pesan en cada clase.

In [ ]:
proba_va = mejor.predict_proba(X_va)
clases = mejor.classes_
conf_va = proba_va.max(axis=1)

errores = pd.DataFrame({"texto": X_va.values, "real": y_va.values,
                        "predicha": pred_va, "confianza": conf_va.round(3)})
errores = errores[errores.real != errores.predicha].sort_values("confianza", ascending=False)
print(f"Errores en validación: {len(errores)} de {len(y_va)}")
errores.head(10)

In [ ]:
vec_pal = mejor.named_steps["feats"].transformer_list[0][1]
n_pal = len(vec_pal.get_feature_names_out())
coef = mejor.named_steps["clf"].coef_[:, :n_pal]
vocab = vec_pal.get_feature_names_out()

filas = coef if len(clases) > 2 else np.vstack([-coef[0], coef[0]])
for i, c in enumerate(clases):
    top = np.argsort(filas[i])[-12:][::-1]
    print(f"[{c}] " + ", ".join(vocab[top]))

**Para discutir:** ¿los términos más influyentes tienen sentido para el negocio? Si aparecen nombres de lugares o de marcas, el modelo puede estar aprendiendo **atajos** del corpus y no la opinión.

## 7. De la etiqueta al *insight*: detección de aspectos
La gerencia no pregunta "¿cuántas reseñas son negativas?", sino **"¿de qué se quejan?"**. Implementamos un detector de aspectos basado en reglas (léxico + expresiones regulares). Es una línea base transparente; en la frontera actual se usa *Aspect-Based Sentiment Analysis* con Transformers o LLM.

> El léxico es un punto de partida: amplíalo revisando reseñas reales y mide su cobertura.

In [ ]:
ASPECTOS = {
    "limpieza":     r"limpi|suci|aseo|polvo|olor|higien|ba[ñn]o",
    "servicio":     r"servicio|atenci|personal|amabl|grosero|recepci|mesero|gu[ií]a",
    "precio":       r"precio|caro|barat|costo|cobr|tarifa|valor|dinero|plata",
    "ubicacion":    r"ubicaci|ubicad|cerca|lejos|acceso|centro|transporte|vista",
    "comida":       r"comida|desayun|almuerz|cena|restaurante|plato|sabor|delicios",
    "instalaciones":r"habitaci|cama|piscina|wifi|internet|instalacion|ruido|ducha|aire",
    "experiencia":  r"paisaje|tour|recorrido|experiencia|visita|actividad|cultura|historia",
}
PATRONES = {a: re.compile(p) for a, p in ASPECTOS.items()}

def detectar_aspectos(texto):
    t = normalizar(texto)
    return [a for a, pat in PATRONES.items() if pat.search(t)]

aspectos_va = X_va.map(detectar_aspectos)
cobertura = (aspectos_va.str.len() > 0).mean()
print(f"Cobertura: {cobertura:.1%} de las reseñas menciona al menos un aspecto del léxico")

filas = [(a, p) for asp, p in zip(aspectos_va, pred_va) for a in asp]
assert filas, "Ningún aspecto detectado: amplía el léxico ASPECTOS."
matriz = pd.crosstab(pd.Series([f[0] for f in filas], name="aspecto"),
                     pd.Series([f[1] for f in filas], name="clase predicha"))
display(matriz)
matriz.plot.barh(stacked=True, figsize=(8, 4), colormap="Blues")
plt.title("Aspectos mencionados por clase predicha"); plt.xlabel("menciones")
plt.tight_layout(); plt.show()

## 8. Generación de respuestas automáticas
Dos estrategias, de menor a mayor riesgo:

1. **Plantillas condicionadas** (clase + aspectos): predecibles, auditables, sin alucinaciones.
2. **LLM local open source** con *prompt* estructurado: más naturales, pero pueden inventar compromisos. Recuerda el caso **Moffatt v. Air Canada (2024)**: la empresa respondió legalmente por lo que dijo su chatbot.

Primero mapeamos cada clase a una **polaridad** para escoger el tono. El mapeo se intenta deducir del nombre de la clase; corrígelo si no coincide con tu dataset.

In [ ]:
NUMERICAS = all(re.fullmatch(r"\d+(\.\d+)?", str(c)) for c in clases)
BINARIA_01 = NUMERICAS and set(map(float, clases)) <= {0.0, 1.0}

def inferir_polaridad(clase):
    c = normalizar(str(clase))
    if BINARIA_01:                                # etiquetas 0/1
        return "positiva" if float(c) == 1 else "negativa"
    if NUMERICAS:                                 # calificaciones tipo 1..5
        v = float(c); return "negativa" if v <= 2 else ("neutra" if v == 3 else "positiva")
    if re.search(r"neg|malo|mala|queja|0$", c): return "negativa"
    if re.search(r"pos|bueno|buena|1$", c):     return "positiva"
    return "neutra"

POLARIDAD = {c: inferir_polaridad(c) for c in clases}
POLARIDAD   # ← revisa y corrige manualmente si hace falta

In [ ]:
NOMBRE_ASPECTO = {"limpieza": "la limpieza", "servicio": "la atención de nuestro equipo",
                  "precio": "la relación calidad-precio", "ubicacion": "la ubicación",
                  "comida": "la comida", "instalaciones": "las instalaciones",
                  "experiencia": "la experiencia y las actividades"}

def unir(items):
    items = [NOMBRE_ASPECTO[a] for a in items]
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " y " + items[-1]

def respuesta_plantilla(texto, clase):
    pol, asp = POLARIDAD.get(clase, "neutra"), detectar_aspectos(texto)
    menc = f" sobre {unir(asp)}" if asp else ""
    if pol == "positiva":
        return f"¡Gracias por tu reseña! Nos alegra que hayas valorado{menc or ' tu visita'}. Te esperamos de nuevo."
    if pol == "negativa":
        return (f"Lamentamos tu experiencia{menc}. Compartimos tu comentario con el área responsable "
                "y un integrante de nuestro equipo se comunicará contigo.")
    return f"Gracias por tus comentarios{menc}. Tus observaciones nos ayudan a mejorar."

muestra = pd.DataFrame({"texto": X_va.values[:6], "clase_predicha": pred_va[:6]})
muestra["respuesta"] = [respuesta_plantilla(t, c) for t, c in zip(muestra.texto, muestra.clase_predicha)]
muestra

Observa que la plantilla negativa **no promete** reembolsos, descuentos ni plazos: cualquier compromiso queda en manos de una persona.

In [ ]:
PROMPT_SISTEMA = (
    "Eres el asistente de atención al cliente de un operador turístico. "
    "Redactas respuestas breves (máximo 3 oraciones), corteses y en español neutro a reseñas de viajeros. "
    "Restricciones: no prometas reembolsos, descuentos, compensaciones ni plazos; "
    "no inventes políticas; si la reseña describe un problema grave, indica que una persona del equipo lo contactará."
)

def prompt_usuario(texto, clase, aspectos):
    return (f"Reseña: \"{texto}\"\n"
            f"Clasificación del sistema: {clase} (polaridad {POLARIDAD.get(clase, 'neutra')})\n"
            f"Aspectos detectados: {', '.join(aspectos) or 'ninguno'}\n"
            "Redacta la respuesta.")

# Guardarraíl: si la respuesta contiene compromisos, se envía a revisión humana
COMPROMISOS = re.compile(r"reembols|devoluci|descuento|compensa|gratis|garantiz|te devolvemos|en \d+ (dias|horas)")
def requiere_revision(respuesta):
    return bool(COMPROMISOS.search(normalizar(respuesta)))

if USE_LLM:
    from transformers import pipeline
    gen = pipeline("text-generation", model=LLM_ID)
    for t, c in zip(muestra.texto[:3], muestra.clase_predicha[:3]):
        msgs = [{"role": "system", "content": PROMPT_SISTEMA},
                {"role": "user", "content": prompt_usuario(t, c, detectar_aspectos(t))}]
        r = gen(msgs, max_new_tokens=120, do_sample=False)[0]["generated_text"][-1]["content"]
        print("RESEÑA:   ", t[:160])
        print("RESPUESTA:", r)
        print("→ revisión humana" if requiere_revision(r) else "→ publicable tras muestreo", "\n")
else:
    print("Sección LLM omitida (USE_LLM = False). Prueba del guardarraíl:")
    for r in ["Gracias, te contactaremos pronto.", "Te ofrecemos un descuento del 20% en tu próxima visita."]:
        print(f"  '{r}' → revisión humana: {requiere_revision(r)}")

## 9. Humano en el bucle: ¿cuánto automatizar?
El sistema **no** tiene que decidir todo. Con un umbral de confianza τ:
- si la confianza ≥ τ → se automatiza (clasificación + respuesta);
- si no → la revisa una persona.

Subir τ reduce errores automatizados pero también la cobertura. Esta curva es **la conversación con el negocio**.

In [ ]:
taus = np.round(np.arange(0.30, 0.96, 0.05), 2)
filas = []
for tau in taus:
    auto = conf_va >= tau
    cov = auto.mean()
    acc_auto = (pred_va[auto] == y_va.values[auto]).mean() if auto.any() else np.nan
    filas.append({"tau": tau, "cobertura_auto": cov, "accuracy_auto": acc_auto})
curva = pd.DataFrame(filas)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(curva.tau, curva.cobertura_auto, "o-", color="#0B1F5C", label="% automatizado")
ax.plot(curva.tau, curva.accuracy_auto, "s-", color="#E07A1F", label="accuracy en lo automatizado")
ax.set_xlabel("umbral de confianza τ"); ax.set_ylim(0, 1.02); ax.grid(alpha=.3); ax.legend()
ax.set_title("Compromiso cobertura vs. calidad")
plt.tight_layout(); plt.show()
curva.round(3)

## 10. Análisis de impacto (costo, eficiencia, experiencia)
Traducimos la curva a lenguaje de negocio. **Los supuestos son ilustrativos**: reemplázalos con datos de la organización (volumen real, tiempos medidos, costo por hora).

In [ ]:
# ---- SUPUESTOS (edítalos) ----
RESENAS_MES          = 3000     # reseñas recibidas por mes
MIN_MANUAL           = 4.0      # minutos para leer, clasificar y responder una reseña a mano
MIN_SUPERVISION      = 0.5      # minutos para revisar por muestreo una reseña automatizada
COSTO_HORA_COP       = 25000    # costo por hora de un agente (COP)
TAU_ELEGIDO          = 0.70

fila = curva.iloc[(curva.tau - TAU_ELEGIDO).abs().argmin()]
auto_mes   = RESENAS_MES * fila.cobertura_auto
manual_mes = RESENAS_MES - auto_mes
horas_antes   = RESENAS_MES * MIN_MANUAL / 60
horas_despues = (manual_mes * MIN_MANUAL + auto_mes * MIN_SUPERVISION) / 60
errores_auto  = auto_mes * (1 - fila.accuracy_auto)

print(f"Umbral τ = {fila.tau:.2f}")
print(f"Reseñas automatizadas/mes: {auto_mes:,.0f}  ({fila.cobertura_auto:.0%})")
print(f"Horas de trabajo/mes:       {horas_antes:,.0f} → {horas_despues:,.0f}")
print(f"Ahorro estimado/mes:        ${(horas_antes - horas_despues) * COSTO_HORA_COP:,.0f} COP")
print(f"Errores automatizados/mes:  {errores_auto:,.0f}  ← el costo oculto que el negocio debe aceptar o mitigar")

**Experiencia del usuario:** el ahorro de horas solo vale si el viajero recibe respuestas oportunas y correctas. Recuerda el caso **Klarna (2024–2025)**: optimizar costo y velocidad degradó la calidad percibida y la empresa volvió a contratar personas. Y la evidencia de **Brynjolfsson et al. (QJE, 2025)** sugiere que el mayor beneficio aparece cuando la IA **asiste** al agente, no cuando lo reemplaza.

## 11. Reflexión crítica (entregable)
Responde en tu informe del proyecto integrador, con evidencia de este notebook:

1. **Valor:** ¿qué decisión de la organización cambia gracias al sistema? ¿Cómo lo medirías después del despliegue?
2. **Baseline vs. Transformer:** ¿la mejora justifica el costo computacional y de mantenimiento?
3. **Errores:** ¿qué tipo de reseña falla más? ¿Hay patrones (ironía, negación, reseñas mixtas, regionalismos)?
4. **Aspectos:** ¿qué cobertura tiene el léxico? ¿Qué aspectos faltan? ¿Cómo lo reemplazarías por un modelo?
5. **Respuestas:** ¿qué controles evitan que el sistema prometa lo que la empresa no puede cumplir?
6. **Umbral τ:** ¿qué valor recomendarías y por qué? ¿Quién debe aprobar esa decisión?
7. **Responsabilidad:** sesgos (variedades del español, turistas extranjeros), privacidad de los autores de reseñas, deriva de los datos con el tiempo.

### Lista de chequeo técnica antes de presentar
- [ ] Partición estratificada y sin fuga de información.
- [ ] Baseline reportado junto al modelo final.
- [ ] Macro-F1 y reporte por clase, no solo *accuracy*.
- [ ] Análisis de errores con ejemplos concretos.
- [ ] Métrica de negocio y supuestos explícitos.
- [ ] Limitaciones y riesgos documentados.

### Referencias
- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed., borrador). Cap. 4: Naive Bayes, clasificación de texto y sentimiento.
- Brynjolfsson, E., Li, D. & Raymond, L. (2025). Generative AI at Work. *The Quarterly Journal of Economics*, 140(2), 889–942. https://doi.org/10.1093/qje/qjae044
- *Moffatt v. Air Canada*, 2024 BCCRT 149 (British Columbia Civil Resolution Tribunal).